<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/Demo_SwissEphemeris_Usage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](http://1.bp.blogspot.com/_5QlanosqxiQ/SXLOLTu7S7I/AAAAAAAAAm8/0r6C7lCxsic/S1600-R/p21-header-003.png) <br>


---

Website - [Parashar21](https://parashar21.blogspot.com)

#Install Swiss Ephemeris <br>
## This is a demonstration of the usage of the Swiss Ephemeris Software in python on the Google Colab Platform
https://pypi.org/project/pyswisseph/  - Python Package Documentation <br>
https://www.astro.com/swisseph/swephinfo_e.htm <br>
https://www.astro.com/swisseph/swephprg.htm - Programmer Documentation <br>
https://pythonhosted.org/pyswisseph/swisseph-module.html

In [1]:
!pip install pyswisseph > /dev/null

In [2]:
# ---------------------------------------------------------------
# SWISS EPHEMERIS DATA FILES -- WHY DO WE NEED THESE?
#
# Swiss Ephemeris calculates planetary positions using external
# binary ephemeris files (.se1), which contain astronomical data.
#
# sepl_18.se1 : Planetary positions (Sun, Mercury, Venus, Mars,
#               Jupiter, Saturn, etc.)
# semo_18.se1 : Lunar positions (Moon)
# seas_18.se1 : Asteroid positions (NOT used by Parashar21)
#
# The _18 files cover approximately 1800-2399 CE.
#
# IMPORTANT:
# These files are NOT mandatory. If the required files are absent,
# Swiss Ephemeris normally falls back to its built-in Moshier
# calculation engine (within Moshier's supported date range).
# The program will normally continue without crashing.
#
# Moshier is sufficiently accurate for most horoscope calculations,
# but small differences in planetary longitudes are possible.
#
# WHY KEEP THE FILES?
# Parashar21 was developed and tested using Swiss Ephemeris data.
# Retaining these files preserves computational consistency.
#
# The files are downloaded into ./ephe and their location is set
# using swe.set_ephe_path('/content/ephe') in p21swe.py.
#
# NOTE:
# seas_18.se1 is unnecessary unless asteroid calculations are added.
# The ephemeris engine actually used can be identified from the
# return flags of swe.calc_ut().
#
# FUTURE MAINTAINER (INCLUDING MY FUTURE SELF):
# Removing these downloads may not break the program, but could
# silently change its astronomical calculation engine.
# ---------------------------------------------------------------

In [3]:
#Download Swiss Ephemeris Files
#!wget  http://www.astro.com/ftp/swisseph/ephe/seas_18.se1
!wget -q https://github.com/aloistr/swisseph/raw/master/ephe/seas_18.se1
#!wget  http://www.astro.com/ftp/swisseph/ephe/semo_18.se1
!wget -q https://github.com/aloistr/swisseph/raw/master/ephe/semo_18.se1
#!wget  http://www.astro.com/ftp/swisseph/ephe/sepl_18.se1
!wget  -q https://github.com/aloistr/swisseph/raw/master/ephe/sepl_18.se1

!mkdir ephe
!mv *.se1 ephe

In [4]:
import swisseph as swe
swe.version

'2.10.03'

In [5]:
#Constants and Flags necessary for this application are set here
#
swe.set_ephe_path('/content/ephe') # set path to ephemeris files
# Calendar : Julian or Gregorian
SE_GREG_CAL = 1
gregflag = SE_GREG_CAL
#
# Ayanamsha type : Lahiri = 1
swe.set_sid_mode(1)  # Lahiri Aynamsha
#
# whether speed will be calculated along with position of planet
SEFLG_SPEED = int(256)
iflag = SEFLG_SPEED
#hsys = P, Placidus
#ascii P = 080
hsysP =  bytes('P', 'utf-8')

#Personal Data
## Data Input

In [6]:
from datetime import datetime
import pytz

In [7]:
# Current time in Asia/Kolkata
now = datetime.now(pytz.timezone("Asia/Kolkata"))

# Convert to UTC
now_utc = now.astimezone(pytz.utc)

# Extract date-time components
year   = now_utc.year
month  = now_utc.month
day    = now_utc.day
hour   = now_utc.hour
minute = now_utc.minute
second = now_utc.second + now_utc.microsecond / 1e6  # keep fractions of seconds

# Convert to Julian Day
jd_ut, jd_et = swe.utc_to_jd(year, month, day, hour, minute, second, swe.GREG_CAL)

print("UTC datetime:", now_utc)
print("Julian Day (UT):", jd_ut)
print("Julian Day (ET):", jd_et)


UTC datetime: 2026-11-04 09:04:40.773701+00:00
Julian Day (UT): 2461348.8790504364
Julian Day (ET): 2461348.878254012


In [8]:
# Date & Time of Birth
# Convert UTC date to Julian Date
#JD = swe.utc_to_jd(1950,9,17,11,0,0,gregflag)
JD = swe.utc_to_jd(year, month, day, hour, minute, second, gregflag)
print (JD[0], JD[1])
# JD[0] is Ephemeris Time
# JD[1] is Universal Time
natalUT = JD[1]
print(natalUT, 'Julian Date in Universal Time')
natalLON = 72.36
natalLAT = 23.48

2461348.8790504364 2461348.878254012
2461348.878254012 Julian Date in Universal Time


In [9]:
#Ayanamsha
ayanamsha = swe.get_ayanamsa(natalUT)
print('Lahiri Ayanamsha :', ayanamsha)

ayanamsha = swe.get_ayanamsa_ut(natalUT)
print('Lahiri Ayanamsha :', ayanamsha)

Lahiri Ayanamsha : 24.232060906452887
Lahiri Ayanamsha : 24.232060936915445


#Calculations

## Without setting SIDEREAL flag

In [10]:
#Position of Planets without SETTING SEFLG_SIDEREAL
#
# body 0 = Sun, 1 = Moon, ... 11 = True Node (Rahu)
bLon = []
bRet = []
for body in [0,1,2,3,4,5,6,11]:
    pData = swe.calc_ut(natalUT,body,iflag)
    #print(pData[0])                    --- this is a major change
    #print(pData[1])                    --- this is a major change
    bLon.append(pData[0][0])            # note this correction
    if pData[0][3] >= 0:                # note this correction
        bRet.append('False')
    else :
        bRet.append('True')
for ix in range(len(bLon)) : print(bLon[ix], bRet[ix])

221.9585049546078 False
163.76293143471068 False
222.47126626425222 True
204.7503220958047 True
140.3041940797492 False
144.7203531898532 False
9.080198367710993 True
326.91760207448823 True


In [11]:
#House Position and Ascendants without setting SEFLG_SIDEREAL
# the 13th value is the longitude of Ascendant
#
P = swe.houses(natalUT,natalLAT,natalLON,hsysP)
for ix in range(len(P[0])) : print(P[0][ix])
print('---')
for ix in range(len(P[1])) : print(P[1][ix])
print('---')
print('Asc Long ',P[1][0])

336.3820318500912
15.082369311654288
47.163079949684935
73.55001322174297
97.88473832311007
124.0217539020444
156.38203185009115
195.08236931165428
227.16307994968494
253.55001322174297
277.88473832311007
304.0217539020444
---
336.3820318500912
253.55001322174297
252.16094695276462
170.2831971657299
340.6713031645919
343.6785750612095
262.14384754604123
163.67857506120944
---
Asc Long  336.3820318500912


## With SIDEREAL flag set

In [12]:
#Position of Planetsd using SEFLG_SIDEREAL flag
#
# body 0 = Sun, 1 = Moon, ... 11 = True Node (Rahu)
bLonS = []
bRetS = []
print(iflag)
SEFLG_SIDEREAL = int(64*1024)      #// sidereal positions
iflag |= SEFLG_SIDEREAL
print(iflag)
for body in [0,1,2,3,4,5,6,11]:
    pData = swe.calc_ut(natalUT,body,iflag)
    #print(pData[0])                    --- this is a major change
    #print(pData[1])                    --- this is a major change
    bLonS.append(pData[0][0])            # note this correction
    if pData[0][3] >= 0:                # note this correction
        bRetS.append('False')
    else :
        bRetS.append('True')
print("Change in Longitude for each planet")
for ix in range(len(bLonS)) : print( bLonS[ix],bRetS[ix], bLonS[ix]-bLon[ix] )

256
65792
Change in Longitude for each planet
197.72416300968263 False -24.234341944925177
139.5285894897855 False -24.234341944925177
198.23692431932702 True -24.234341944925205
180.5159801508795 True -24.234341944925205
116.06985213482405 False -24.234341944925163
120.48601124492802 False -24.234341944925177
344.8458564227858 True 335.7656580550748
302.68326012956305 True -24.234341944925177


In [13]:
#House Position and Ascendants without setting SEFLG_SIDEREAL
# the 13th value is the longitude of Ascendant
#
hsysPb = hsysP.decode("utf-8")
print(hsysPb)
PS = swe.houses_ex(natalUT,natalLAT,natalLON,hsysP,iflag)   # the documentation is WRONG, iflag must be the last parameter, not the second
#for ix in range(len(PS[0])) : print(PS[0][ix])
for ix in range(len(PS[1])) : print(PS[1][ix], PS[1][ix]-P[1][ix])
#for ix in range(len(PS[1])) : print(PS[1][0]-P[1][0])

P
312.14768990516603 -24.234341944925177
229.31567127681782 -24.23434194492515
252.16094695276462 0.0
146.04885522080474 -24.23434194492515
316.4369612196667 -24.234341944925177
319.4442331162843 -24.234341944925177
237.90950560111608 -24.23434194492515
139.4442331162843 -24.23434194492515


In [14]:
!date

Wed Nov  4 09:04:40 AM UTC 2026


The output generated here matches with the output generated by the Windows program swetest

#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [Finished scrolling? Now try reading a book. ](http://chronos.yantrajaal.com) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)